# UC4_Executive_Business_Intelligence

```
=============================================================================
UC4 — EXECUTIVE BUSINESS INTELLIGENCE (BONUS)
=============================================================================
Reads from : dbx_retail.gold.mv_revenue_by_region
             dbx_retail.gold.mv_return_rate_by_vendor
             dbx_retail.gold.mv_slow_moving_products
             dbx_retail.gold.fact_orders
             dbx_retail.gold.fact_returns
             dbx_retail.gold.dim_vendors
Writes to  : dbx_retail.gold.ai_business_insights
             dbx_retail.gold.pipeline_run_log
Model      : databricks-gpt-oss-20b via Databricks AI Gateway
=============================================================================
DESIGN RULE — NEVER pass raw rows to the LLM.
Aggregate first (SUM, AVG, COUNT, distribution by region).
Pass 5 KPI values. A well-designed prompt with 5 KPIs produces a better
executive summary than a prompt with 5000 raw rows — and won't hit context limits.

OUTPUT DOMAINS:
  1. revenue_performance      — revenue by region, MoM comparison
  2. vendor_return_rate       — vendor return rates, highest risk vendors
  3. slow_moving_inventory    — slow-moving products by region, dollar exposure
=============================================================================
```

In [0]:
%pip install openai
dbutils.library.restartPython()

In [0]:
import time
import json
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from openai import OpenAI

client = OpenAI(
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
    base_url="https://4092199907892374.ai-gateway.cloud.databricks.com/mlflow/v1"
)

MODEL_NAME    = "databricks-gpt-oss-20b"
NOTEBOOK_NAME = "UC4_Executive_Business_Intelligence"

print("LLM client ready.")
print(f"Run started at : {datetime.now().isoformat()}")

In [0]:

def extract_text(response) -> str:
    content = response.choices[0].message.content
    if isinstance(content, list):
        parts = [
            block["text"]
            for block in content
            if isinstance(block, dict) and block.get("type") == "text"
        ]
        return " ".join(parts).strip()
    return content.strip()


def call_llm(prompt: str, system_msg: str, retries: int = 3, wait: int = 2) -> str:
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user",   "content": prompt}
                ],
                max_tokens=2000,
                temperature=0.4   # slightly higher for narrative variety
            )
            return extract_text(response)
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(wait * (attempt + 1))
            else:
                return f"LLM_UNAVAILABLE: {str(e)}"


def validate_llm_output(text: str, min_words: int = 40) -> dict:
    refusal_phrases = ["i cannot", "as an ai", "i don't have", "i'm unable"]
    issues = []
    if len(text.split()) < min_words:
        issues.append("too_short")
    if text.startswith("LLM_UNAVAILABLE"):
        issues.append("llm_call_failed")
    if any(p in text.lower() for p in refusal_phrases):
        issues.append("refusal_detected")
    return {
        "text":         text,
        "quality_flag": "PASS" if not issues else "REVIEW",
        "issues":       ", ".join(issues) if issues else "none"
    }


SYSTEM_MSG_EXEC = (
    "You are a senior business analyst writing executive briefings for the CEO "
    "and regional directors of dbx_retail, a national retail chain. "
    "Write in plain, authoritative business English. No bullet points, no markdown, no headers. "
    "4-6 sentences of connected prose. Name specific regions, vendors, or figures where provided."
)

print("Helpers defined.")

In [0]:

df_revenue    = spark.table("dbx_retail.gold.mv_revenue_by_region")
df_vendor_rr  = spark.table("dbx_retail.gold.mv_return_rate_by_vendor")
df_slow       = spark.table("dbx_retail.gold.mv_slow_moving_products")
df_orders     = spark.table("dbx_retail.gold.fact_orders")
df_returns    = spark.table("dbx_retail.gold.fact_returns")
df_vendors    = spark.table("dbx_retail.gold.dim_vendors")

print("Gold Materialized Views loaded:")
print(f"  mv_revenue_by_region        : {df_revenue.count()} rows")
print(f"  mv_return_rate_by_vendor    : {df_vendor_rr.count()} rows")
print(f"  mv_slow_moving_products     : {df_slow.count()} rows")

In [0]:
# Compute: total revenue by region (current vs prior period), top/bottom region,
# MoM delta. NEVER pass raw order rows to the LLM.

# Current period = most recent order_month in the data
latest_month = df_revenue.agg(F.max("order_month")).collect()[0][0]

revenue_current = (
    df_revenue
    .filter(F.col("order_month") == latest_month)
    .groupBy("region")
    .agg(
        F.sum("total_sales").alias("total_sales"),
        F.sum("order_count").alias("order_count")
    )
    .orderBy(F.col("total_sales").desc())
    .collect()
)

# Prior period for MoM comparison
revenue_prior = (
    df_revenue
    .filter(F.col("order_month") < latest_month)
    .groupBy("region")
    .agg(F.sum("total_sales").alias("prior_sales"))
    .collect()
)

prior_by_region = {r["region"]: r["prior_sales"] for r in revenue_prior}

revenue_kpis = []
for r in revenue_current:
    region       = r["region"]
    current_val  = round(float(r["total_sales"] or 0), 2)
    prior_val    = round(float(prior_by_region.get(region, current_val) or current_val), 2)
    mom_delta    = round(((current_val - prior_val) / prior_val * 100) if prior_val > 0 else 0, 1)
    revenue_kpis.append({
        "region":       region,
        "total_sales":  current_val,
        "order_count":  int(r["order_count"] or 0),
        "mom_delta_pct": mom_delta
    })

total_revenue  = round(sum(r["total_sales"] for r in revenue_kpis), 2)
top_region     = revenue_kpis[0]["region"] if revenue_kpis else "N/A"
bottom_region  = revenue_kpis[-1]["region"] if revenue_kpis else "N/A"

revenue_kpi_json = json.dumps({
    "period":          str(latest_month),
    "total_revenue":   total_revenue,
    "top_region":      top_region,
    "bottom_region":   bottom_region,
    "by_region":       revenue_kpis
})

print(f"Revenue KPIs computed for period: {latest_month}")
print(f"Total revenue: ${total_revenue:,.2f}")
for r in revenue_kpis:
    print(f"  {r['region']:<10} : ${r['total_sales']:>12,.2f}  | MoM: {r['mom_delta_pct']:+.1f}%")

In [0]:

revenue_prompt = f"""dbx_retail's regional revenue performance for {latest_month} is summarized below.

KPI DATA:
{revenue_kpi_json}

Write a 4-6 sentence executive briefing for the CEO that covers:
1. Overall revenue performance for the period
2. Which region led and which region underperformed, with specific figures
3. The month-over-month trend — which regions grew and which declined
4. One strategic observation or recommendation based on the data

Plain business English. No bullet points. No markdown."""

revenue_summary_raw  = call_llm(revenue_prompt, SYSTEM_MSG_EXEC)
revenue_summary      = validate_llm_output(revenue_summary_raw)

print("\nREVENUE PERFORMANCE SUMMARY:")
print("=" * 70)
print(revenue_summary["text"])
print(f"\nQuality flag: {revenue_summary['quality_flag']}")

In [0]:
vendor_rr_agg = (
    df_vendor_rr
    .drop("vendor_name")                                          # remove duplicate before join
    .join(df_vendors.select("vendor_id", "vendor_name"), on="vendor_id", how="left")
    .groupBy("vendor_id", "vendor_name")
    .agg(
        F.avg("return_rate").alias("avg_return_rate"),
        F.sum("total_returns").alias("total_returns"),
        F.sum("total_orders").alias("total_orders")
    )
    .withColumn("return_rate_pct", F.round(F.col("avg_return_rate") * 100, 1))
    .orderBy(F.col("avg_return_rate").desc())
    .collect()
)

high_risk_vendors = [
    r for r in vendor_rr_agg if float(r["avg_return_rate"] or 0) > 0.20
]
avg_return_rate_overall = round(
    sum(float(r["avg_return_rate"] or 0) for r in vendor_rr_agg) / max(len(vendor_rr_agg), 1) * 100, 1
)

vendor_kpi_json = json.dumps({
    "total_vendors_analyzed":    len(vendor_rr_agg),
    "vendors_above_20pct":       len(high_risk_vendors),
    "avg_return_rate_overall":   avg_return_rate_overall,
    "top_5_highest_return_rate": [
        {
            "vendor":      r["vendor_name"] or r["vendor_id"],
            "return_rate": float(r["return_rate_pct"] or 0)
        }
        for r in vendor_rr_agg[:5]
    ]
})

print(f"\nVendor return rate KPIs — {len(vendor_rr_agg)} vendors analyzed")
print(f"Vendors above 20% return rate: {len(high_risk_vendors)}")
for r in vendor_rr_agg[:5]:
    print(f"  {str(r['vendor_name'] or r['vendor_id']):<30} : {r['return_rate_pct']}%")

In [0]:

vendor_prompt = f"""dbx_retail vendor return rate analysis is summarized below.

KPI DATA:
{vendor_kpi_json}

Write a 4-6 sentence executive briefing for the regional merchandise director that covers:
1. The overall return rate picture across vendors
2. Which specific vendors are above the acceptable 20% threshold, with their rates
3. The financial risk and operational implication of high-return vendors
4. A recommended next step for the merchandise team

Plain business English. No bullet points. No markdown."""

vendor_summary_raw = call_llm(vendor_prompt, SYSTEM_MSG_EXEC)
vendor_summary     = validate_llm_output(vendor_summary_raw)

print("\nVENDOR RETURN RATE SUMMARY:")
print("=" * 70)
print(vendor_summary["text"])
print(f"\nQuality flag: {vendor_summary['quality_flag']}")

In [0]:

slow_agg = (
    df_slow
    .groupBy("region")
    .agg(
        F.count("*").alias("slow_product_count"),
        F.sum(
            F.when(F.col("unit_price").isNotNull(), F.col("unit_price")).otherwise(0)
        ).alias("estimated_inventory_exposure")
    )
    .orderBy(F.col("slow_product_count").desc())
    .collect()
)

total_slow_products = sum(int(r["slow_product_count"] or 0) for r in slow_agg)
total_exposure      = round(sum(float(r["estimated_inventory_exposure"] or 0) for r in slow_agg), 2)
worst_region        = slow_agg[0]["region"] if slow_agg else "N/A"

slow_kpi_json = json.dumps({
    "total_slow_moving_products":   total_slow_products,
    "total_inventory_exposure_usd": total_exposure,
    "worst_performing_region":      worst_region,
    "by_region": [
        {
            "region":      r["region"],
            "slow_count":  int(r["slow_product_count"] or 0),
            "exposure_usd": round(float(r["estimated_inventory_exposure"] or 0), 2)
        }
        for r in slow_agg
    ]
})

print(f"\nSlow-moving inventory KPIs")
print(f"Total slow-moving products: {total_slow_products}")
print(f"Estimated inventory exposure: ${total_exposure:,.2f}")
for r in slow_agg:
    print(f"  {r['region']:<10} : {r['slow_product_count']} products")

In [0]:

slow_prompt = f"""dbx_retail's slow-moving inventory analysis is summarized below.

KPI DATA:
{slow_kpi_json}

Write a 4-6 sentence executive briefing for the inventory and merchandising director that covers:
1. The scale of the slow-moving inventory problem across regions
2. Which region has the highest concentration of slow-moving products
3. The estimated financial exposure tied up in unsold inventory
4. A recommended action to address the inventory blindspot

Plain business English. No bullet points. No markdown."""

slow_summary_raw = call_llm(slow_prompt, SYSTEM_MSG_EXEC)
slow_summary     = validate_llm_output(slow_summary_raw)

print("\nSLOW-MOVING INVENTORY SUMMARY:")
print("=" * 70)
print(slow_summary["text"])
print(f"\nQuality flag: {slow_summary['quality_flag']}")

In [0]:

insights = [
    {
        "insight_type":   "revenue_performance",
        "ai_summary":     revenue_summary["text"],
        "kpi_json":       revenue_kpi_json,
        "quality_flag":   revenue_summary["quality_flag"],
        "quality_issues": revenue_summary["issues"],
        "generated_at":   datetime.now().isoformat()
    },
    {
        "insight_type":   "vendor_return_rate",
        "ai_summary":     vendor_summary["text"],
        "kpi_json":       vendor_kpi_json,
        "quality_flag":   vendor_summary["quality_flag"],
        "quality_issues": vendor_summary["issues"],
        "generated_at":   datetime.now().isoformat()
    },
    {
        "insight_type":   "slow_moving_inventory",
        "ai_summary":     slow_summary["text"],
        "kpi_json":       slow_kpi_json,
        "quality_flag":   slow_summary["quality_flag"],
        "quality_issues": slow_summary["issues"],
        "generated_at":   datetime.now().isoformat()
    }
]

schema = StructType([
    StructField("insight_type",   StringType(), True),
    StructField("ai_summary",     StringType(), True),
    StructField("kpi_json",       StringType(), True),
    StructField("quality_flag",   StringType(), True),
    StructField("quality_issues", StringType(), True),
    StructField("generated_at",   StringType(), True)
])

df_insights = spark.createDataFrame(insights, schema=schema)

(
    df_insights.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("dbx_retail.gold.ai_business_insights")
)

print("\nInsights written to dbx_retail.gold.ai_business_insights")
display(spark.table("dbx_retail.gold.ai_business_insights"))

In [0]:
# Databricks SQL-native function for calling Foundation Models inside a query.
# Demonstrates at least 2 working examples on real Gold layer data.
# Run these cells in a SQL notebook or as %sql magic cells.

print("""
COPY AND RUN THE FOLLOWING IN A DATABRICKS SQL NOTEBOOK OR %sql CELL:
=====================================================================

-- DEMO 1: ai_query() on vendor return rates
-- Generates a one-sentence plain-English interpretation per vendor

SELECT
    vendor_id,
    vendor_name,
    return_rate_pct,
    ai_query(
        'databricks-gpt-oss-20b',
        CONCAT(
            'In one sentence, explain whether a return rate of ',
            CAST(return_rate_pct AS STRING),
            '% for vendor ', vendor_name,
            ' at dbx_retail retail is acceptable, concerning, or critical. ',
            'Plain English only. No markdown.'
        )
    ) AS vendor_risk_interpretation
FROM dbx_retail.gold.mv_return_rate_by_vendor
ORDER BY return_rate_pct DESC
LIMIT 10;


-- DEMO 2: ai_query() on revenue trend
-- Generates a one-sentence trend commentary per region

SELECT
    region,
    total_sales,
    order_count,
    ai_query(
        'databricks-gpt-oss-20b',
        CONCAT(
            'dbx_retail region "', region, '" generated $',
            CAST(ROUND(total_sales, 0) AS STRING),
            ' across ', CAST(order_count AS STRING), ' orders. ',
            'In one sentence, state whether this performance signals growth, ',
            'stability, or concern for a retail chain of this scale. ',
            'Plain English only. No markdown.'
        )
    ) AS revenue_commentary
FROM dbx_retail.gold.mv_revenue_by_region
WHERE order_month = (SELECT MAX(order_month) FROM dbx_retail.gold.mv_revenue_by_region)
ORDER BY total_sales DESC;

=====================================================================
""")

In [0]:

review_count = sum(
    1 for s in [revenue_summary, vendor_summary, slow_summary]
    if s["quality_flag"] == "REVIEW"
)

run_log = [{
    "notebook":          NOTEBOOK_NAME,
    "run_timestamp":     datetime.now().isoformat(),
    "records_processed": 3,     # 3 insight domains
    "llm_calls_made":    3,
    "outputs_flagged":   review_count,
    "status":            "SUCCESS" if review_count == 0 else "PARTIAL_REVIEW",
    "notes":             f"domains=revenue,vendor_return_rate,slow_inventory | period={latest_month}"
}]

(
    spark.createDataFrame(run_log)
    .write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("dbx_retail.gold.pipeline_run_log")
)

print(f"Run log written. Status: {run_log[0]['status']}")
print(f"Summaries requiring review: {review_count} / 3")